# 1. Extract Coordinates from TIFF Image

This notebook will help extract:
- Geolocation coordinates (latitude/longitude)
- Date and time information
- Image metadata

from your TIFF image of the oil spill.

In [1]:
# Install required packages if needed
import sys
import subprocess

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

required_packages = ['rasterio', 'Pillow']
for package in required_packages:
    try:
        __import__(package)
        print(f"{package} is already installed")
    except ImportError:
        print(f"Installing {package}...")
        install_package(package)

rasterio is already installed
Installing Pillow...


In [2]:
# Import required libraries
import rasterio
from PIL import Image
from PIL.ExifTags import TAGS
import os
import datetime
import json

def get_image_metadata(image_path):
    """Extract all metadata from image file"""
    metadata = {}
    
    # Get TIFF metadata using rasterio
    with rasterio.open(image_path) as src:
        metadata['bounds'] = src.bounds._asdict()
        metadata['crs'] = str(src.crs)
        metadata['transform'] = list(src.transform)
        metadata['shape'] = src.shape
    
    # Get EXIF metadata using PIL
    try:
        with Image.open(image_path) as img:
            exif = img._getexif()
            if exif:
                for tag_id in exif:
                    tag = TAGS.get(tag_id, tag_id)
                    data = exif.get(tag_id)
                    # Decode bytes if needed
                    if isinstance(data, bytes):
                        try:
                            data = data.decode()
                        except:
                            data = str(data)
                    metadata[tag] = data
    except:
        print("No EXIF metadata found")
    
    return metadata

# You can use this function once you place your TIFF file in the data/raw folder
# Example usage:
# metadata = get_image_metadata('../data/raw/your_image.tif')
# print(json.dumps(metadata, indent=2))

In [3]:

# Specify your image path here
image_path = '../data/raw/2023-04-23-00_00_2023-04-23-23_59_Sentinel-1_IW_VV+VH_VV_-_decibel_gamma0.tiff'  # Change this to your image path

import glob, json, os
from pprint import pprint

raw_dir = os.path.join('..', 'data', 'raw')

# If image_path not provided, auto-detect TIFF in data/raw
if not os.path.exists(image_path):
    patterns = ['*.tif', '*.tiff']
    files = []
    for p in patterns:
        files.extend(glob.glob(os.path.join(raw_dir, p)))

    if not files:
        print('No TIFF files found in', raw_dir)
        print('Please set image_path variable above to your image location')
    else:
        print('Found TIFF(s):')
        for f in files:
            print(' -', f)
        # pick first file
        image_path = files[0]
        print('\nUsing:', image_path)
else:
    print('Using specified path:', image_path)

if os.path.exists(image_path):
    print('\nExtracting metadata from:', image_path)
    try:
        metadata = get_image_metadata(image_path)
        # Print key fields
        print('\n--- Extracted metadata summary ---')
        pprint({
            'bounds': metadata.get('bounds'),
            'crs': metadata.get('crs'),
            'shape': metadata.get('shape'),
            'transform': metadata.get('transform'),
            'DateTime': metadata.get('DateTime') or metadata.get('datetime') or metadata.get('TIFFTAG_DATETIME')
        })
        # Save to processed folder for downstream use
        out_json = os.path.join('..', 'data', 'processed', os.path.basename(image_path) + '.metadata.json')
        with open(out_json, 'w') as fh:
            json.dump(metadata, fh, default=str, indent=2)
        print('\nSaved metadata to', out_json)
    except Exception as e:
        print('Failed to extract metadata:', e)



Using specified path: ../data/raw/2023-04-23-00_00_2023-04-23-23_59_Sentinel-1_IW_VV+VH_VV_-_decibel_gamma0.tiff

Extracting metadata from: ../data/raw/2023-04-23-00_00_2023-04-23-23_59_Sentinel-1_IW_VV+VH_VV_-_decibel_gamma0.tiff
No EXIF metadata found

--- Extracted metadata summary ---
{'DateTime': None,
 'bounds': {'bottom': 13.200171608892623,
            'left': 121.40304167281266,
            'right': 121.69555265914079,
            'top': 13.400305854588545},
 'crs': 'EPSG:4326',
 'shape': (1757, 2500),
 'transform': [0.00011700439453125,
               0.0,
               121.40304167281266,
               0.0,
               -0.00011390679891629058,
               13.400305854588545,
               0.0,
               0.0,
               1.0]}

Saved metadata to ..\data\processed\2023-04-23-00_00_2023-04-23-23_59_Sentinel-1_IW_VV+VH_VV_-_decibel_gamma0.tiff.metadata.json


In [4]:

# Specify your image path here
image_path = '../data/raw/2023-03-18-00_00_2023-03-18-23_59_Sentinel-1_IW_VV+VH_VV_-_decibel_gamma0(zoomed).tiff'  # Change this to your image path

import glob, json, os
from pprint import pprint

raw_dir = os.path.join('..', 'data', 'raw')

# If image_path not provided, auto-detect TIFF in data/raw
if not os.path.exists(image_path):
    patterns = ['*.tif', '*.tiff']
    files = []
    for p in patterns:
        files.extend(glob.glob(os.path.join(raw_dir, p)))

    if not files:
        print('No TIFF files found in', raw_dir)
        print('Please set image_path variable above to your image location')
    else:
        print('Found TIFF(s):')
        for f in files:
            print(' -', f)
        # pick first file
        image_path = files[0]
        print('\nUsing:', image_path)
else:
    print('Using specified path:', image_path)

if os.path.exists(image_path):
    print('\nExtracting metadata from:', image_path)
    try:
        metadata = get_image_metadata(image_path)
        # Print key fields
        print('\n--- Extracted metadata summary ---')
        pprint({
            'bounds': metadata.get('bounds'),
            'crs': metadata.get('crs'),
            'shape': metadata.get('shape'),
            'transform': metadata.get('transform'),
            'DateTime': metadata.get('DateTime') or metadata.get('datetime') or metadata.get('TIFFTAG_DATETIME')
        })
        # Save to processed folder for downstream use
        out_json = os.path.join('..', 'data', 'processed', os.path.basename(image_path) + '.metadata.json')
        with open(out_json, 'w') as fh:
            json.dump(metadata, fh, default=str, indent=2)
        print('\nSaved metadata to', out_json)
    except Exception as e:
        print('Failed to extract metadata:', e)


Using specified path: ../data/raw/2023-03-18-00_00_2023-03-18-23_59_Sentinel-1_IW_VV+VH_VV_-_decibel_gamma0(zoomed).tiff

Extracting metadata from: ../data/raw/2023-03-18-00_00_2023-03-18-23_59_Sentinel-1_IW_VV+VH_VV_-_decibel_gamma0(zoomed).tiff
No EXIF metadata found

--- Extracted metadata summary ---
{'DateTime': None,
 'bounds': {'bottom': 13.245946984164917,
            'left': 121.39068911028639,
            'right': 121.6832000966145,
            'top': 13.446043383717733},
 'crs': 'EPSG:4326',
 'shape': (1757, 2500),
 'transform': [0.00011700439453124431,
               0.0,
               121.39068911028639,
               0.0,
               -0.00011388525870962777,
               13.446043383717733,
               0.0,
               0.0,
               1.0]}

Saved metadata to ..\data\processed\2023-03-18-00_00_2023-03-18-23_59_Sentinel-1_IW_VV+VH_VV_-_decibel_gamma0(zoomed).tiff.metadata.json
